# Camada Bronze — Datajud Raw

Lê os arquivos JSON gerados pelo script de ingestão (M2) e salva como tabela Delta
no Unity Catalog sem nenhuma transformação. Dados brutos preservados exatamente
como vieram da API.

**Tabela de destino:** `judicial.bronze.datajud_raw`

**Pré-requisito:** fazer upload dos JSONs do M2 para o DBFS:
```
dbfs:/FileStore/bronze/datajud/TJSC/
dbfs:/FileStore/bronze/datajud/TJPR/
```

## 2. Parâmetros

In [0]:
VOLUME_BASE = "/Volumes/judicial/bronze/raw_files"
TABELA       = "judicial.bronze.datajud_raw"
TRIBUNAIS    = ["TJSC", "TJPR"]

## 3. Ingestão

Cada arquivo JSON do M2 é um array de hits do Elasticsearch.
Com `multiline=true` o Spark trata cada elemento do array como uma linha.

Colunas adicionadas:
- `tribunal` — identifica a origem (necessário porque lemos tribunal por tribunal)
- `data_ingestao` — timestamp de quando o dado entrou no lake

In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit

for tribunal in TRIBUNAIS:
    path = f"{VOLUME_BASE}/{tribunal}/"

    try:
        files = dbutils.fs.ls(path)
        if not files:
            print(f"{tribunal}: nenhum arquivo encontrado, pulando")
            continue
    except Exception as e:
        print(f"{tribunal}: erro ao acessar caminho — {e}")
        continue

    df = (
        spark.read
        .option("multiline", "true")
        .json(path)
    )

    df = (
        df
        .withColumn("tribunal", lit(tribunal))
        .withColumn("data_ingestao", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .partitionBy("tribunal")
        .saveAsTable(TABELA)
    )

    # Move todos os arquivos processados — itera sobre a lista já buscada,
    # evitando hardcode de nome e uma segunda chamada ao fs.
    processados_path = f"{VOLUME_BASE}/processados/{tribunal}/"
    for file_info in files:
        dest = f"{processados_path}{file_info.name}"
        dbutils.fs.mv(file_info.path, dest)

    count = df.count()
    print(f"{tribunal}: {count:,} processos gravados em {TABELA}, {len(files)} arquivo(s) arquivado(s)")

## 4. Verificação

In [0]:
%sql
SELECT
    tribunal,
    COUNT(*)          AS total_processos,
    MIN(data_ingestao) AS primeira_ingestao,
    MAX(data_ingestao) AS ultima_ingestao
FROM judicial.bronze.datajud_raw
GROUP BY tribunal
ORDER BY tribunal;

tribunal,total_processos,primeira_ingestao,ultima_ingestao
TJPR,1,2026-05-25T19:59:56.856Z,2026-05-25T19:59:56.856Z


In [0]:
%sql
SELECT * FROM judicial.bronze.datajud_raw LIMIT 10;